In [ ]:
#Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend
!pip install -q google-genai pydantic
from google import genai
from google.genai import types
import json
from pydantic import BaseModel, Field
from google.colab import userdata

import pandas as pd




# Enter API Key
API_KEY = userdata.get("Gemini_API_Key")

client = genai.Client(api_key=API_KEY)

# Define the model
MODEL_NAME = "gemini-3.8-flash"

# Define the schema
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

# Write the system Instructions
SYSTEM_TEXT = """
You are an appointment no-show analysis assistant.

Analyse the patient's appointment note carefully.
Identify the most likely reason for missing the appointment.
Use only information supported by the note.
Do not invent facts.
Return only valid JSON matching the supplied schema.
"""

# Build the CRAFT
def build_prompt(note_text):

    prompt = f"""
Context:
We are analysing patient notes from a medical appointment
no-show dataset to understand why patients missed appointments.

Role:
Act as a healthcare appointment operations analyst.

Action:
Read the patient's note and identify the most likely root cause.
Choose only one of these values:

- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other

Provide a confidence score between 0 and 1.
Write a summary of no more than 25 words.
Recommend one practical action for the clinic.

Tone:
Be neutral, clear, professional and concise.
Do not make assumptions that are not supported by the note.

Format:
Return exactly four fields:
root_cause, confidence, summary, recommended_action.

Patient note:
{note_text}
"""
    return prompt


# Create analyse_note(note_text)
import time
import json

def analyse_note(note_text, return_raw=False):

    prompt = build_prompt(note_text)

    max_retries = 5

    for attempt in range(max_retries):

        try:
            print(f"Gemini attempt {attempt + 1}")

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA
                )
            )

            raw_response = response.text
            data = json.loads(raw_response)

            if return_raw:
                return data, raw_response

            return data

        except Exception as e:

            if attempt < max_retries - 1:

                wait_time = 10 * (attempt + 1)

                print("Gemini temporarily unavailable.")
                print(f"Retrying after {wait_time} seconds.")

                time.sleep(wait_time)

            else:
                print("Gemini failed after all attempts.")
                raise

import pandas as pd

notes_df = pd.read_csv("/content/followup_notes.csv")

print(notes_df.head())
print(notes_df.shape)
print(notes_df.columns)

note_n001 = notes_df.loc[
notes_df["note_id"] == "N001",
      "note_text"
     ].iloc[0]
result = analyse_note(note_n001)

print(result)



In [ ]:
# Task 4.2 — Validate and process the 10 notes

#Define the Pydantic model

from pydantic import BaseModel, Field
import json
import time
from google.genai.errors import ServerError

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# API call counter
import time
import json

api_calls = 0

def analyse_note(note_text, return_raw=False):
    global api_calls
    prompt = build_prompt(note_text)
    max_retries = 3

    for attempt in range(max_retries):
        try:
            api_calls += 1
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA
                )
            )
            raw_response = response.text

            try:
                data = json.loads(raw_response)
            except (json.JSONDecodeError, TypeError):
                data = None

            if return_raw:
                return data, raw_response
            return data

        except ServerError as e:
            if e.code == 503 and attempt < max_retries - 1:
                sleep_time = (attempt + 1) * 10
                print(
                    f"Gemini API 503 (Unavailable) error: {e.message}. "
                    f"Retrying in {sleep_time} seconds (Attempt {attempt + 1}/{max_retries})..."
                )
                time.sleep(sleep_time)
            else:
                raise e
        except Exception as e:
            if attempt < max_retries - 1:
                sleep_time = (attempt + 1) * 10
                print(
                    f"API error: {e}. "
                    f"Retrying in {sleep_time} seconds..."
                )
                time.sleep(sleep_time)
            else:
                raise RuntimeError(
                    f"Gemini failed after {max_retries} attempts: {e}"
                )

#Read the CSV and inspect the first 10 rows
import pandas as pd

notes_df = pd.read_csv("/content/followup_notes.csv")

print(notes_df.head(10))
print(notes_df.shape)
print(notes_df.columns)

# Loop through the first 10 rows
results = []

for index, row in notes_df.head(10).iterrows():
    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    result = {
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Check missing or blank notes first
    if pd.isna(note_text) or str(note_text).strip() == "":
        result["root_cause"] = "other"
        result["confidence"] = 0.0
        result["summary"] = "No note recorded"
        result["recommended_action"] = "Re-call patient"
        result["validation_status"] = "skipped_blank"
        print(note_id, "Skipped: blank note")
    else:
        try:
            # Pause before each API call after the first
            if api_calls > 0:
                time.sleep(13)

            data, raw_response = analyse_note(
                str(note_text),
                return_raw=True
            )

            # Preserve original response
            result["raw_response"] = raw_response

            if not isinstance(data, dict):
                raise ValueError(
                    "Response is not a valid JSON object"
                )

            # Copy only the fields actually returned
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:
                if field in data:
                    result[field] = data[field]

            # Pydantic validation
            insight = FollowupInsight(**data)

            # Check allowed root cause
            if insight.root_cause not in ALLOWED_ROOT_CAUSES:
                raise ValueError("Invalid root_cause value")

            result["validation_status"] = "valid"

        except Exception as e:
            result["validation_status"] = "invalid"
            print(note_id, "Invalid response:", str(e))

    results.append(result)

print("Total API calls:", api_calls)

# Create and save the output CSV
output_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

notes_analysed_df = pd.DataFrame(
    results,
    columns=output_columns
)

notes_analysed_df.to_csv(
    "notes_analysed.csv",
    index=False
)

print(notes_analysed_df)

# Check dimensions
print("Shape:", notes_analysed_df.shape)

# Check all 10 notes are present
print("Number of notes:", len(notes_analysed_df))

# Check validation status
print("\nValidation status:")
print(notes_analysed_df["validation_status"].value_counts())

# Check N004
print("\nN004:")
print(
    notes_analysed_df[
        notes_analysed_df["note_id"] == "N004"
    ].to_string(index=False)
)

# Check confidence values
valid_rows = notes_analysed_df[
    notes_analysed_df["validation_status"].isin(
        ["valid", "skipped_blank"]
    )
]

print("\nConfidence within 0-1:")
print(valid_rows["confidence"].between(0, 1).all())

expected = {
    "N001": "forgot_or_no_reminder",
    "N002": "distance_or_transport",
    "N003": "cost_or_fee",
    "N005": "illness_or_emergency",
    "N006": "work_or_schedule",
    "N007": "forgot_or_no_reminder",
    "N008": "distance_or_transport",
    "N009": "other",
    "N010": "work_or_schedule"
}

comparison = notes_analysed_df[
    notes_analysed_df["note_id"].isin(expected)
].copy()

comparison["expected"] = comparison["note_id"].map(expected)

comparison["agreement"] = (
    comparison["root_cause" ] == comparison["expected"]
)

print(comparison[
    ["note_id", "root_cause", "expected",
     "agreement", "validation_status"]
])

print("\nAgreement:", comparison["agreement"].sum(), "out of 9")

In [ ]:
#  4.3 — Break it on purpose

# A – Test Pydantic validation

from pydantic import ValidationError

# Bad reply 1: confidence is above the allowed maximum
bad_reply_1 = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "Unclear reason for absence",
    "recommended_action": "Call the patient"
}

# Bad reply 2: recommended_action is missing
bad_reply_2 = {
    "root_cause": "other",
    "confidence": 0.5,
    "summary": "Unclear reason for absence"
}

# Test the first reply
try:
    FollowupInsight(**bad_reply_1)
except ValidationError as e:
    print("Bad reply 1 - Validation error:")
    print(e)

# Test the second reply
try:
    FollowupInsight(**bad_reply_2)
except ValidationError as e:
    print("\nBad reply 2 - Validation error:")
    print(e)

    # B – Compare V1 and V2 prompts

    note_ids = ["N007", "N009"]

selected_notes = notes_df[
    notes_df["note_id"].isin(note_ids)
]

for _, row in selected_notes.iterrows():
    print(row["note_id"], ":", row["note_text"])

    # define the two prompts.

    def build_v1_prompt(note_text):

      return f"""
Classify this note:

{note_text}
"""


def build_v2_prompt(note_text):
    return f"""
Context:
We are analysing patient notes to understand why appointments were missed.

Role:
Act as an appointment operations analyst.

Action:
Read the note and select the most likely root cause.

Choose exactly one of these allowed values:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other

If the note is ambiguous or does not provide enough information,
choose "other" rather than guessing.

Provide:
- root_cause
- confidence between 0 and 1
- summary of no more than 25 words
- one practical recommended_action

Tone:
Be neutral, clear and concise. Do not invent facts.

Format:
Return only JSON matching the supplied schema.

Patient note:
{note_text}
"""

# use the same schema and settings for both prompts.

task43_calls = 0

def compare_prompt(prompt):

    global task43_calls

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    task43_calls += 1

    raw_response = response.text

    try:
        data = json.loads(raw_response)
    except (json.JSONDecodeError, TypeError):
        data = None

    return data, raw_response

    # run the comparison for both notes

comparison_results = []

for _, row in selected_notes.iterrows():

    note_id = row["note_id"]
    note_text = str(row["note_text"])

    prompts = {
        "V1": build_v1_prompt(note_text),
        "V2": build_v2_prompt(note_text)
    }

    for version, prompt in prompts.items():

        data, raw_response = compare_prompt(prompt)

        if isinstance(data, dict):
            root_cause = data.get("root_cause", "")

            allowed = root_cause in ALLOWED_ROOT_CAUSES
        else:
            root_cause = ""
            allowed = False

        comparison_results.append({
            "note_id": note_id,
            "prompt": version,
            "root_cause": root_cause,
            "allowed_value": allowed,
            "raw_response": raw_response
        })

        # Pause between API calls
        if not (
            note_id == "N009" and version == "V2"
        ):
            time.sleep(13)

    # Result display

    comparison_df = pd.DataFrame(comparison_results)

print(
    comparison_df[
        ["note_id", "prompt", "root_cause", "allowed_value"]
    ].to_string(index=False)
)

# C – Count API calls across Tasks 4.1–4.3

# Task 4.1: one call, assuming no retries
calls_task41 = 1

# Task 4.2: api_calls is the counter from your earlier loop
calls_task42 = api_calls

# Task 4.3: four comparison calls
calls_task43 = task43_calls

total_api_calls = (
    calls_task41 +
    calls_task42 +
    calls_task43
)

print("Task 4.1 calls:", calls_task41)
print("Task 4.2 calls:", calls_task42)
print("Task 4.3 calls:", calls_task43)
print("Total API calls:", total_api_calls)


In [ ]:
# Task 4.4 — Recommendation and GitHub

# A1 – Merge the two CSV files

import pandas as pd

# Read both files
notes_analysed_df = pd.read_csv("/content/notes_analysed.csv")
appointments_df = pd.read_csv("/content/appointments_clean.csv")

# Merge using appointment_id
merged_df = pd.merge(
    notes_analysed_df,
    appointments_df[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left"
)

print("Merged shape:", merged_df.shape)
print(merged_df.head(10))

# A2 – Cross-tabulate root cause against reminder status

cause_reminder_table = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print(cause_reminder_table)

# Check forgot/no-reminder notes
forgot_notes = merged_df[
    merged_df["root_cause"] == "forgot_or_no_reminder"
]

print("Forgot/no-reminder notes:")
print(forgot_notes[["note_id", "reminder_sent"]])

print(
    "Number with reminder_sent = No:",
    (forgot_notes["reminder_sent"] == "No").sum()
)

# Check total notes with no reminder
print(
    "Total notes with reminder_sent = No:",
    (merged_df["reminder_sent"] == "No").sum()
)

